# Lab 3: Structured Output

**Difficulty: Beginner | ~30 min | Requires Lab 1 (recommended)**

A model answers in words; a program needs data. **Structured output** is the bridge: you define the exact shape you want back (a schema), and the model returns a real, typed object that matches it — no markdown fences, no commentary, no parsing by hand. You will first see how messy free-form text really is, then define a schema with Pydantic, get a typed object back with one method call, and finally build a tiny pipeline that turns a stack of plain-text reviews into data you can compute on.

## Step 1

One command installs all required modules, versions pinned so the lab is reproducible.

In [ ]:
# One command installs all required modules (versions pinned for reproducibility)
!pip install "langchain==1.2.15" "langchain-core==1.2.28" "langchain-openai==1.1.12" "python-dotenv==1.2.2" "pydantic==2.13.4"


## Step 2

Loads your OpenRouter API key from `.env` and stops with a clear message if it's missing.

In [ ]:
import os
from dotenv import load_dotenv

# Read the OPENROUTER_API_KEY we saved in .env (Section 9 of the guide)
load_dotenv()

# Stop early with a clear message if the key is missing
if not os.getenv("OPENROUTER_API_KEY"):
    raise SystemExit("No OPENROUTER_API_KEY found. Add it to .env and restart the kernel.")

## Step 3

Create the chat model — the same wrapper and settings you used in Labs 1 and 2.

In [ ]:
from langchain_openai import ChatOpenAI

# The model: same wrapper and settings as Lab 1
model = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",  # a free model on OpenRouter
    base_url="https://openrouter.ai/api/v1",   # redirect the OpenAI client to OpenRouter
    api_key=os.getenv("OPENROUTER_API_KEY"),   # your key, read from .env
    temperature=0,                              # 0 = factual, reproducible
)

## Step 4

The problem: ask for JSON in plain English and see what actually comes back.

In [ ]:
# Ask for JSON in plain English — no schema, no enforcement
reply = model.invoke(
    "In one sentence, describe the movie 'Inception', then return a JSON object "
    "about it with fields title, director, and year."
)

print(reply.content)
print(f"type: {type(reply.content).__name__}")

## Step 5

Define the shape we want back — a Pydantic schema — instead of hoping for clean JSON.

In [ ]:
from pydantic import BaseModel, Field

# A schema: the exact shape we want the model's answer to take.
# Each field has a type (str or int) and a description telling the model
# what to put there. This class is the contract we hand to the model.
class Movie(BaseModel):
    title: str = Field(description="The movie's title")
    director: str = Field(description="The director's full name")
    year: int = Field(description="The movie's release year")

## Step 6

Wrap the model so its output must match the schema — `invoke` now returns a real `Movie` object, not text.

In [ ]:
# with_structured_output binds the schema to the model: every reply is
# parsed and validated into a Movie object before you see it.
structured_model = model.with_structured_output(Movie)

movie = structured_model.invoke(
    "Return structured details for 'Inception' by Christopher Nolan, released in 2010."
)

print(f"type:     {type(movie).__name__}")
print(f"title:    {movie.title}")
print(f"director: {movie.director}")
print(f"year:     {movie.year}")

## Step 7

The output is data, not text — a real Python object you can inspect and pass around.

In [ ]:
# A Movie is a real Python object, so we can treat it as data:
print(movie.model_dump())                 # as a plain dict
print(movie.model_dump_json(indent=2))    # as a clean JSON string
print(isinstance(movie, Movie))           # it really is a Movie

## Step 8

Now a realistic job: pull structured facts out of a plain-text customer review.

In [ ]:
# A new schema for a new job: extracting facts from a review
class ProductReview(BaseModel):
    product: str = Field(description="The exact product being reviewed")
    rating: int = Field(description="The star rating, from 1 to 5")
    sentiment: str = Field(description="positive, negative, or neutral")


review_text = (
    "I bought the 'AeroPress Coffee Maker' two weeks ago and it makes the best "
    "cup of coffee. The whole process takes about two minutes and cleanup is "
    "effortless. Five stars from me."
)

# The same with_structured_output trick, one line: text in, ProductReview out
review = model.with_structured_output(ProductReview).invoke(review_text)
print(review.model_dump())

## Step 9

The payoff: many reviews in, typed records out, and code that computes on the data.

In [ ]:
# Three short reviews, one per line of the list
reviews_text = [
    "I bought the 'AeroPress Coffee Maker' two weeks ago and it makes the best cup of coffee. Five stars from me.",
    "The 'Ergonomic Office Chair' arrived broken and customer service never replied. One star, terrible.",
    "My 'Wireless Noise-Cancelling Headphones' are great value — comfortable and the battery lasts all week. Four stars.",
]

# Turn each review into a typed ProductReview object
parsed_reviews = []
for review_text in reviews_text:
    parsed = model.with_structured_output(ProductReview).invoke(review_text)
    parsed_reviews.append(parsed)
    print(f"{parsed.product} | rating {parsed.rating}/5 | {parsed.sentiment}")

# Because the results are typed data, we can compute on them directly —
# no string parsing, no cleaning up markdown fences.
average_rating = sum(r.rating for r in parsed_reviews) / len(parsed_reviews)
print(f"Average rating: {average_rating:.1f}")

## Optional Exercise

Swap the model. In a new cell, create a second `ChatOpenAI` with a different free model (any model listed as `:free` at https://openrouter.ai/models — for example `nvidia/nemotron-3-nano-30b-a3b:free`), build `structured_model_2 = model_2.with_structured_output(Movie)`, and ask it the same "Inception" question from Step 6. Confirm you get back a `Movie` object with title `Inception`, director `Christopher Nolan`, and year `2010` — proving the schema constraint holds no matter which model is behind it.